# Abschnitt 3: Klassifikation von Münzen

Dieses Notebook ist für **Google Colab** gedacht. Es lädt CSV-Dateien aus dem Dashboard, extrahiert Peak-Merkmale und vergleicht drei Klassifikationsverfahren:

1. manuelle Grenzen
2. k-Nearest Neighbors (k-NN)
3. Entscheidungsbaum

Am Ende wird eine `model.json` erzeugt, die im Dashboard auf den ESP32 hochgeladen werden kann.

## 1. Bibliotheken importieren

Wir verwenden `pandas` für Tabellen, `numpy` für Rechnungen und `scikit-learn` für die Klassifikatoren. Diese Pakete sind in Google Colab normalerweise bereits installiert.

In [ ]:
from __future__ import annotations

import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURES = ["rp_min", "l_min", "delta_rp", "delta_l"]

# Servo-Zielwinkel für den Live-Test am Sortierer.
# Diese Werte müssen an den mechanischen Aufbau angepasst werden.
CLASS_ANGLES = {
    "1ct": 25,
    "2ct": 35,
    "5ct": 45,
    "10ct": 65,
    "20ct": 75,
    "50ct": 85,
    "1eu": 110,
    "2eu": 125,
}

print("Setup fertig")

## 2. CSV-Dateien hochladen

Lade hier die CSV-Dateien aus dem Dashboard hoch. Sinnvolle Dateinamen sind zum Beispiel:

- `muenze_1ct.csv`
- `muenze_10ct.csv`
- `muenze_1eu.csv`

Das Label wird aus dem Dateinamen abgeleitet.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_FILES = {name: data for name, data in uploaded.items() if name.lower().endswith(".csv")}
except Exception:
    # Fallback für lokale Jupyter-Umgebungen:
    # CSV-Dateien in den Ordner messdaten/ legen.
    CSV_FILES = {p.name: p.read_bytes() for p in Path("messdaten").glob("*.csv")}

print(f"{len(CSV_FILES)} CSV-Dateien geladen")
list(CSV_FILES.keys())[:10]

## 3. Hilfsfunktionen

Die nächsten Funktionen übernehmen drei Aufgaben:

- Label aus Dateinamen bestimmen
- CSV-Dateien einlesen
- Peaks finden und daraus Merkmale berechnen

Die Peak-Erkennung ist bewusst einfach gehalten: Ein Peak beginnt, wenn `RP` deutlich unter die Luftreferenz fällt, und endet, wenn das Signal zurückkehrt.

In [ ]:
def label_from_filename(filename: str) -> str:
    """Leitet das Klassenlabel aus dem Dateinamen ab."""
    stem = Path(filename).stem.lower()
    for token in ("muenze_", "munze_", "coin_"):
        stem = stem.replace(token, "")
    return stem


def read_dashboard_csv(raw: bytes) -> pd.DataFrame:
    """Liest Dashboard-CSV ein. Neuere Exporte nutzen Semikolon, alte ggf. Komma."""
    text = raw.decode("utf-8-sig")
    df = pd.read_csv(io.StringIO(text), sep=";")
    if len(df.columns) == 1:
        df = pd.read_csv(io.StringIO(text))
    df = df.rename(columns=str.strip)
    required = {"Zeit_ms", "RP", "L"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV-Spalten fehlen: {missing}")
    return df


def extract_peaks(df: pd.DataFrame, label: str, min_gap: int = 3) -> list[dict]:
    """Extrahiert Peak-Features aus einer Messreihe."""
    rp = df["RP"].to_numpy(dtype=float)
    l_val = df["L"].to_numpy(dtype=float)

    # Robuste Luftreferenz: hoher RP-Wert und niedriger L-Wert entsprechen typischerweise Luft.
    rp_air = float(np.percentile(rp, 90))
    l_air = float(np.percentile(l_val, 90))

    # Ein Objekt wird angenommen, wenn RP deutlich unter die Luftreferenz fällt.
    baseline_window = rp[: min(len(rp), 50)]
    noise = max(float(np.std(baseline_window)), 20.0)
    threshold = rp_air - 4.0 * noise

    peaks = []
    active = False
    start = 0
    gap = 0

    for i, value in enumerate(rp):
        present = value < threshold

        if present and not active:
            active = True
            start = i
            gap = 0
        elif active and not present:
            gap += 1
            if gap >= min_gap:
                end = max(start + 1, i - gap + 1)
                seg_rp = rp[start:end]
                seg_l = l_val[start:end]

                # Sehr kurze Segmente werden ignoriert, weil sie meist Rauschen sind.
                if len(seg_rp) >= 3:
                    rp_min = float(np.min(seg_rp))
                    l_min = float(np.min(seg_l))
                    peaks.append({
                        "label": label,
                        "rp_min": rp_min,
                        "l_min": l_min,
                        "delta_rp": rp_air - rp_min,
                        "delta_l": l_air - l_min,
                    })
                active = False
        elif active:
            gap = 0

    return peaks

## 4. Peaks extrahieren

Aus jeder CSV-Datei werden jetzt Peaks extrahiert. Die resultierende Tabelle enthält eine Zeile pro Münzdurchlauf.

In [ ]:
rows = []
raw_tables = {}

for filename, raw in CSV_FILES.items():
    label = label_from_filename(filename)
    df = read_dashboard_csv(raw)
    raw_tables[filename] = df
    rows.extend(extract_peaks(df, label))

data = pd.DataFrame(rows)

print(f"{len(data)} Peaks extrahiert")
display(data.head())
display(data.groupby("label").size().rename("Peaks pro Klasse"))

## 5. Streudiagramm der Merkmale

Das Streudiagramm hilft, manuelle Grenzen abzuleiten und zu erkennen, welche Klassen gut oder schlecht trennbar sind.

In [ ]:
plt.figure(figsize=(7, 5))
for label, group in data.groupby("label"):
    plt.scatter(group["delta_rp"], group["delta_l"], label=label, alpha=0.8)

plt.xlabel("delta_rp")
plt.ylabel("delta_l")
plt.title("Peak-Features der Münzen")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Trainings- und Testdaten bilden

Wir teilen die Daten in Trainingsdaten und Testdaten. Das Modell wird nur mit den Trainingsdaten gebaut. Die Testdaten bleiben zurück, um zu prüfen, wie gut das Modell auf neuen Messungen funktioniert.

In [ ]:
X = data[FEATURES].to_numpy()
y = data["label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# k-NN und Entscheidungsbaum arbeiten besser vergleichbar, wenn alle Features skaliert sind.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training: {len(X_train)} Peaks")
print(f"Test:     {len(X_test)} Peaks")

## 7. Methode 1: Manuelle Grenzen

Die folgenden Regeln sind nur ein Startpunkt. Sie sollen anhand des Streudiagramms angepasst werden. Für vollständige Einzelmünzen müssen die Regeln oft feiner werden als die drei Beispielgruppen.

In [ ]:
def manual_predict_one(row: np.ndarray) -> str:
    """Beispiel-Regeln. Werte an das eigene Streudiagramm anpassen."""
    rp_min, l_min, delta_rp, delta_l = row

    # Beispiel: Stahlmünzen erzeugen oft eine stärkere L-Änderung.
    if delta_l > 80:
        return "stahl"

    # Beispiel: stark leitfähige Münzen erzeugen oft eine große RP-Änderung.
    if delta_rp > 2500 and delta_l < 80:
        return "nordic"

    return "bimetall"


manual_pred = np.array([manual_predict_one(row) for row in X_test])

print("Manuelle Grenzen")
print(classification_report(y_test, manual_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, manual_pred)
plt.title("Konfusionsmatrix: manuelle Grenzen")
plt.xticks(rotation=45)
plt.show()

## 8. Methode 2: k-NN

k-NN klassifiziert eine neue Messung anhand der nächsten Trainingspunkte im Merkmalsraum. Der Parameter `k` bestimmt, wie viele Nachbarn abstimmen.

In [ ]:
for k in (1, 3, 5):
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train_s, y_train)
    print(f"k={k}: Genauigkeit = {knn_tmp.score(X_test_s, y_test):.3f}")

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_s, y_train)
knn_pred = knn.predict(X_test_s)

print(classification_report(y_test, knn_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, knn_pred)
plt.title("Konfusionsmatrix: k-NN")
plt.xticks(rotation=45)
plt.show()

## 9. Methode 3: Entscheidungsbaum

Ein Entscheidungsbaum lernt Wenn-Dann-Regeln. Wir begrenzen die Tiefe auf `max_depth=3`, damit der Baum noch erklärbar bleibt.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train_s, y_train)
tree_pred = tree.predict(X_test_s)

print("Gelernte Baumregeln:")
print(export_text(tree, feature_names=FEATURES))

print(classification_report(y_test, tree_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, tree_pred)
plt.title("Konfusionsmatrix: Entscheidungsbaum")
plt.xticks(rotation=45)
plt.show()

## 10. Modell für ESP32 exportieren

Diese Zelle schreibt `model.json`. Die Datei enthält die Feature-Skalierung, die Trainingspunkte für k-NN und die Knoten des Entscheidungsbaums. Danach kann sie im Dashboard hochgeladen werden.

In [ ]:
def export_tree_nodes(clf: DecisionTreeClassifier) -> list[dict]:
    """Konvertiert den scikit-learn-Baum in ein kompaktes JSON-Format."""
    tree_ = clf.tree_
    labels = clf.classes_
    nodes = []

    for i in range(tree_.node_count):
        is_leaf = tree_.children_left[i] == tree_.children_right[i]

        if is_leaf:
            label = str(labels[int(np.argmax(tree_.value[i][0]))])
            nodes.append({
                "feature": -1,
                "threshold": 0,
                "left": -1,
                "right": -1,
                "label": label,
                "angle": CLASS_ANGLES.get(label, 90),
            })
        else:
            nodes.append({
                "feature": int(tree_.feature[i]),
                "threshold": float(tree_.threshold[i]),
                "left": int(tree_.children_left[i]),
                "right": int(tree_.children_right[i]),
                "label": "",
                "angle": 90,
            })

    return nodes


model = {
    "version": 2,
    "features": FEATURES,
    "classes": [
        {"label": str(label), "angle": CLASS_ANGLES.get(str(label), 90)}
        for label in sorted(set(y))
    ],
    "manual": {"rules": []},
    "knn": {
        "k": 3,
        "mean": scaler.mean_.tolist(),
        "std": scaler.scale_.tolist(),
        "samples": [
            {
                "label": str(label),
                "angle": CLASS_ANGLES.get(str(label), 90),
                "x": x.tolist(),
            }
            for x, label in zip(X_train_s, y_train)
        ],
    },
    "tree": {"nodes": export_tree_nodes(tree)},
}

with open("model.json", "w", encoding="utf-8") as f:
    json.dump(model, f, indent=2)

print("model.json geschrieben")

## 11. `model.json` herunterladen

In Colab lädt die nächste Zelle die Datei direkt herunter. Anschließend kann sie im Dashboard unter `model.json hochladen` an den ESP32 übertragen werden.

In [ ]:
try:
    from google.colab import files
    files.download("model.json")
except Exception:
    print("Lokale Umgebung: model.json liegt im aktuellen Arbeitsordner.")